# Transformers playground

Seven parts. Each one stands alone: run the two setup cells, then jump to what you need.

1. **The GPU.** Whether you have one, and what it changes.
2. **Hugging Face.** Where models come from.
3. **Tokens.** What your text looks like to a model.
4. **ModernBERT.** Fill in a blank. Read a word in context. Watch context arrive, layer by
   layer.
5. **One vector per passage.** CLS, mean, trained. Then search two novels by meaning.
6. **CLIP.** Search paintings by typing. See the patches. See where it looks.
7. **Train a CLIP.** The whole rule, on thirty pictures.

Free Colab runs all of it. Models download once and stay for the session.

> Save a copy first: **File → Save a copy in Drive**. Colab wipes its disk when the runtime ends.

In [ ]:
%pip install -q -U transformers sentence-transformers

In [ ]:
import os, re, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image

from transformers import logging as hf_logging
hf_logging.set_verbosity_error()                 # quieten the load reports

print("torch", torch.__version__)

## 1 · The GPU

A GPU does thousands of multiplications at once. That is all a transformer does.

Colab: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
if torch.cuda.is_available():
    DEVICE = "cuda"
    print("GPU:", torch.cuda.get_device_name(0),
          round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
elif torch.backends.mps.is_available():          # Apple silicon, running locally
    DEVICE = "mps"
    print("Apple GPU")
else:
    DEVICE = "cpu"
    print("No GPU. Everything still works, slower.")
print("device:", DEVICE)

In [ ]:
# One big matrix multiply, timed on each device you have.
def time_matmul(device, n=2000, reps=5):
    a = torch.randn(n, n, device=device)
    b = torch.randn(n, n, device=device)
    if device == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(reps):
        a @ b
    if device == "cuda":
        torch.cuda.synchronize()
    return (time.time() - t0) / reps

cpu = time_matmul("cpu")
print(f"cpu   {cpu*1000:7.0f} ms")
if DEVICE != "cpu":
    gpu = time_matmul(DEVICE)
    print(f"{DEVICE:5s} {gpu*1000:7.0f} ms   ({cpu/gpu:.0f}x faster)")

## 2 · Hugging Face

[huggingface.co](https://huggingface.co/models) is where the models live. Names look like
`answerdotai/ModernBERT-base`: account, then model. Read the **model card** first. It says what
the model was trained on and what its licence allows.

`pipeline` downloads the model, its tokenizer, and the code around both.

In [ ]:
from transformers import pipeline

sentiment = pipeline("sentiment-analysis", device=DEVICE)
for line in ["The sequel was two hours I will never get back.",
             "I have not stopped thinking about the last twenty minutes.",
             "It was, I suppose, a film."]:
    out = sentiment(line)[0]
    print(f"{out['label']:8s} {out['score']:.2f}  {line}")

print("\nyou just used:", sentiment.model.config._name_or_path)

Two problems. You did not pick the model, the cell did. And it was trained on product
reviews. Read the third line and decide whether you agree with the answer.

In [ ]:
# Where the downloads go. Delete this folder to reclaim the disk.
from pathlib import Path
cache = Path(os.environ.get("HF_HOME", Path.home() / ".cache/huggingface"))
print(cache)
if cache.exists():
    size = sum(f.stat().st_size for f in cache.rglob("*") if f.is_file())
    print(f"{size/1e9:.2f} GB cached")

## 3 · Tokens

A model never sees letters. It sees **tokens**: common words whole, rare ones in pieces. That
is why a rare name costs more tokens than a common one.

In [ ]:
from transformers import AutoTokenizer

MODEL = "answerdotai/ModernBERT-base"
tok = AutoTokenizer.from_pretrained(MODEL)

for line in ["I ate the whole pizza.",
             "antidisestablishmentarianism",
             "Nosferatu, Whitby, Bistritz",
             "https://example.com/page?id=42"]:
    pieces = [p.replace("\u0120", " ").strip() for p in tok.tokenize(line)]
    print(f"{len(pieces):3d} | " + " / ".join(pieces))

In [ ]:
# Tokens are numbers. This is the actual input to every model below.
enc = tok("A museum keeps its collection.")
print(enc["input_ids"])
print(tok.convert_ids_to_tokens(enc["input_ids"]))

## 4 · ModernBERT

**ModernBERT** (2024) is a rebuilt BERT: 22 layers, 8,192 tokens of context. It is an
*encoder*. It does not write text. It reads a passage and returns a vector for every token.

It was trained by filling in blanks, and you can still ask it to.

In [ ]:
fill = pipeline("fill-mask", model=MODEL, device=DEVICE)

for s in ["I put the milk in the [MASK].",
          "The best thing about Chicago is the [MASK].",
          "Mary Shelley wrote [MASK] in 1818.",
          "My code failed because I forgot a [MASK]."]:
    print(f"{s:46s} {[g['token_str'].strip() for g in fill(s, top_k=4)]}")

The Shelley line fills the blank with something grammatical, not something true. It learned
which words fit where.

### The same word, four sentences

In [ ]:
from transformers import AutoModel

model = AutoModel.from_pretrained(MODEL).to(DEVICE).eval()

def token_span(sentence, word):
    """Where `word` sits in the token list."""
    enc = tok(sentence, return_tensors="pt")
    ids = enc["input_ids"][0].tolist()
    want = tok(" " + word, add_special_tokens=False)["input_ids"]
    at = [i for i in range(len(ids) - len(want) + 1) if ids[i:i+len(want)] == want]
    if not at:
        raise ValueError(f"'{word}' not found as a whole token in: {sentence}")
    return enc, at[0], len(want)

def word_vector(sentence, word):
    enc, i, n = token_span(sentence, word)
    with torch.no_grad():
        states = model(**enc.to(DEVICE)).last_hidden_state[0]
    v = states[i:i+n].mean(0)
    return (v / v.norm()).cpu().numpy()

sentences = ["A bat flew out of the cave at dusk.",
             "The bat hung upside down all winter.",
             "She swung the bat and missed.",
             "He gripped the bat with both hands."]
V = np.stack([word_vector(s, "bat") for s in sentences])

print("     " + "".join(f"   s{i+1}" for i in range(4)))
for i, row in enumerate(V @ V.T):
    print(f"  s{i+1} " + "".join(f" {x:+.2f}" for x in row) + "   " + sentences[i])

Two animals together, two baseball bats together, the pairs apart. GloVe gives *bat* one
vector, the same one always. This gives it a new one in every sentence.

### Where context arrives

Layer 0 is the lookup table, before any attention. Same word, same vector, exactly. Then each
layer mixes in the neighbours. Watch the two blocks separate.

In [ ]:
def word_layers(sentence, word):
    """The word's vector at every layer: (23, 768), normalised."""
    enc, i, n = token_span(sentence, word)
    with torch.no_grad():
        states = model(**enc.to(DEVICE), output_hidden_states=True).hidden_states
    v = np.stack([h[0][i:i+n].mean(0).float().cpu().numpy() for h in states])
    return v / np.linalg.norm(v, axis=-1, keepdims=True)

L = np.stack([word_layers(s, "bat") for s in sentences])       # 4 sentences, 23 layers, 768
off = ~np.eye(4, dtype=bool)                                   # ignore the diagonal
fig, axes = plt.subplots(1, 4, figsize=(11, 3))
for ax, layer in zip(axes, [0, 7, 15, 22]):
    M = L[:, layer] @ L[:, layer].T
    ax.imshow(np.where(off, M, np.nan), cmap="RdYlBu_r",
              vmin=M[off].min(), vmax=M[off].max())
    ax.set_title(f"layer {layer}", fontsize=10)
    ax.set_xticks(range(4), [f"s{i+1}" for i in range(4)], fontsize=8)
    ax.set_yticks(range(4), [f"s{i+1}" for i in range(4)], fontsize=8)
    for a in range(4):
        for b in range(4):
            if a != b:
                ax.text(b, a, f"{M[a,b]:.2f}", ha="center", va="center", fontsize=7)
fig.suptitle('"bat" compared with itself, four sentences', x=0.02, ha="left")
plt.tight_layout(); plt.show()

Layer 0 is flat: every pair is exactly 1.00. By layer 22 the two animals (s1, s2) and the two
baseball bats (s3, s4) have pulled apart from each other.

Every number is high, even at the end. Raw encoder vectors all point roughly the same way, so
0.94 on its own means nothing. Read the blocks, not the values. Part 5 fixes this.

Averaging over more words makes the pattern steadier.

In [ ]:
# Four words with two senses each, two sentences per sense.
PAIRS = {
 "bank":  [("She sat on the grassy bank of the river.", "We moored against the muddy bank."),
           ("The bank refused the loan.", "He works at the bank on the high street.")],
 "crane": [("A crane waded in the shallows.", "The crane stood on one leg in the marsh."),
           ("The crane lifted a steel beam.", "A crane swung over the building site.")],
 "bat":   [("A bat flew out of the cave at dusk.", "The bat hung upside down all winter."),
           ("She swung the bat and missed.", "He gripped the bat with both hands.")],
 "seal":  [("A seal basked on the rocks.", "The seal dived after the fish."),
           ("The wax seal was still unbroken.", "He pressed his seal into the wax.")],
}
same, diff = [], []
for word, (sense_a, sense_b) in PAIRS.items():
    a1, a2 = (word_layers(s, word) for s in sense_a)
    b1, b2 = (word_layers(s, word) for s in sense_b)
    same.append(((a1*a2).sum(-1) + (b1*b2).sum(-1)) / 2)
    diff.append(((a1*b1).sum(-1) + (a2*b2).sum(-1)) / 2)
same, diff = np.mean(same, 0), np.mean(diff, 0)

plt.figure(figsize=(7, 4))
plt.plot(same, lw=2, color="#2E6E8E", label="same sense")
plt.plot(diff, lw=2, color="#A34526", label="different sense")
plt.fill_between(range(len(same)), diff, same, color="#A34526", alpha=0.12)
plt.axhline(1, color="#999", lw=0.6)
plt.annotate("layer 0: identical", (0, 1), xytext=(1.5, 0.985), fontsize=9,
             arrowprops=dict(arrowstyle="->", lw=0.8, color="#666"))
plt.xlabel("layer"); plt.ylabel("cosine similarity")
plt.title("when the two senses come apart", loc="left")
plt.legend(frameon=False); plt.tight_layout(); plt.show()
gap = same - diff
print(f"gap  layer 0: {gap[0]:+.3f}   layer 2: {gap[2]:+.3f}   last layer: {gap[-1]:+.3f}")

Zero at the input, open by layer 2, widest at the top. Context comes from the layers.

## 5 · One vector per passage

A vector per token is too many. To compare whole passages you need one vector each. Three ways
to get it:

- **CLS** — the first token, which BERT was meant to use as a summary.
- **Mean** — average all the token vectors.
- **A trained sentence model** — an encoder someone fine-tuned so that its output vectors are
  directly comparable.

They are not equally good. The test: 240 passages from *Dracula* and *Frankenstein*. For each
one, find its nearest neighbour. Is it from the same book?

In [ ]:
def passages(path, book, target=120):
    raw = open(path, encoding="utf-8", errors="ignore").read()
    a, b = raw.find("*** START OF"), raw.find("*** END OF")
    body = raw[raw.find("\n", a) + 1:b] if a > 0 else raw
    paras = [" ".join(p.split()) for p in re.split(r"\n\s*\n", body)]
    paras = [p for p in paras if len(p.split()) >= 60]
    step = max(1, len(paras) // target)
    return [(book, p[:900]) for p in paras[::step]][:target]

for base in ("data/texts", "notebooks/data/texts", "../notebooks/data/texts"):
    if os.path.isdir(base):
        break
else:
    raise FileNotFoundError("clone the repo first, or point `base` at your own texts")

rows = (passages(f"{base}/dracula.txt", "Dracula")
        + passages(f"{base}/frankenstein.txt", "Frankenstein"))
books = np.array([b for b, _ in rows])
texts = [t for _, t in rows]
print(len(rows), "passages")
print(texts[3][:120], "...")

In [ ]:
def pool(texts, how, batch=16):
    out = []
    for i in range(0, len(texts), batch):
        enc = tok(texts[i:i+batch], return_tensors="pt", padding=True,
                  truncation=True, max_length=256).to(DEVICE)
        with torch.no_grad():
            h = model(**enc).last_hidden_state
        m = enc["attention_mask"].unsqueeze(-1)
        v = h[:, 0] if how == "cls" else (h * m).sum(1) / m.sum(1)
        out.append(v.float().cpu().numpy())
    V = np.vstack(out)
    return V / np.linalg.norm(V, axis=1, keepdims=True)

from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE)

ways = {
    "CLS, untrained":  pool(texts, "cls"),
    "mean, untrained": pool(texts, "mean"),
    "MiniLM, trained": embedder.encode(texts, normalize_embeddings=True, batch_size=32),
}
for name, V in ways.items():
    S = V @ V.T
    np.fill_diagonal(S, -9)
    right = (books[S.argmax(1)] == books).mean()
    off = S[np.triu_indices(len(V), 1)]
    print(f"{name:17s} neighbour from same book {right:.0%}   "
          f"cosines {off.min():+.2f} to {off.max():+.2f}")

Look at the cosine range, not just the accuracy. The untrained pooling squeezes every pair into
a narrow band near the top: two unrelated paragraphs score 0.96, and so do two near-identical
ones. Nothing readable there. The trained model spreads them out, so the number means
something. That is what a sentence-embedding model is for.

In [ ]:
S = ways["MiniLM, trained"]

def search(question, k=3):
    scores = S @ embedder.encode([question], normalize_embeddings=True)[0]
    for i in np.argsort(-scores)[:k]:
        print(f"  {scores[i]:.2f} {books[i]:13s} {' '.join(texts[i].split()[:16])}...")

for q in ["someone climbs down a wall",
          "a creature asks to be less alone",
          "the weather turns and the ship is in danger",
          "reading a letter from home"]:
    print("\n" + q)
    search(q)

None of those words are in the passages. That is the difference between this and counting.

In [ ]:
search("write your own question here", k=3)

In [ ]:
from sklearn.decomposition import PCA

pts = PCA(n_components=2).fit_transform(S)
plt.figure(figsize=(7, 5))
for book, colour in [("Dracula", "#A34526"), ("Frankenstein", "#2E6E8E")]:
    m = books == book
    plt.scatter(pts[m, 0], pts[m, 1], s=16, color=colour, alpha=0.75, label=book)
plt.xticks([]); plt.yticks([])
plt.legend(frameon=False)
plt.title("240 passages, arranged by meaning", loc="left")
plt.tight_layout(); plt.show()

Nobody told it which book each passage came from. They overlap in the middle, which is
correct: those are the passages that could be from either.

## 6 · CLIP

CLIP was trained on 400 million picture–caption pairs: pull each picture towards its own
caption, push it away from everyone else's. The result is one space holding both, so a sentence
and an image get comparable vectors.

The 18 Met paintings below carry no tags and no descriptions. Only titles, which the model
never sees.

In [ ]:
import csv

for mbase in ("data/week01", "notebooks/data/week01", "../notebooks/data/week01"):
    if os.path.isdir(os.path.join(mbase, "met")):
        break
else:
    raise FileNotFoundError("clone the repo first")

rows = list(csv.DictReader(open(f"{mbase}/met_manifest.csv", encoding="utf-8")))
paths = [os.path.join(mbase, r["file"]) for r in rows]
titles = [r["title"] for r in rows]

clip = SentenceTransformer("clip-ViT-B-32", device=DEVICE)
pictures = clip.encode([Image.open(p) for p in paths], normalize_embeddings=True)
print(len(paths), "paintings,", pictures.shape[1], "numbers each")

In [ ]:
def look_for(phrase, k=3):
    scores = pictures @ clip.encode([phrase], normalize_embeddings=True)[0]
    best = np.argsort(-scores)[:k]
    fig, axes = plt.subplots(1, k, figsize=(3.4 * k, 3.4))
    for ax, i in zip(axes, best):
        ax.imshow(Image.open(paths[i]))
        ax.set_title(f"{scores[i]:.2f}  {titles[i][:28]}", loc="left", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(f'"{phrase}"', x=0.02, ha="left", fontsize=12)
    plt.tight_layout(); plt.show()

look_for("somebody staring straight at you")
look_for("a stormy sky")

### Labels with no training

Write the labels as sentences, embed them, give each painting the nearest one. No examples, no
fitting. Change the list and run it again.

In [ ]:
LABELS = ["a painting of a man", "a painting of a woman", "a landscape",
          "a religious painting", "a still life"]

scores = pictures @ clip.encode(LABELS, normalize_embeddings=True).T
for i, t in enumerate(titles):
    j = int(np.argmax(scores[i]))
    print(f"{LABELS[j]:26s} {scores[i, j]:.2f}   {t[:44]}")

Find Bronzino's *Portrait of a Young Man*. CLIP files it under woman at 0.32, the same score it
gives the ones it gets right. The score is confidence, not correctness.

### What the picture looks like to it

CLIP's image side is a **Vision Transformer**. It cuts the image into a 7×7 grid of 32-pixel
patches, turns each patch into a token, and runs the same machinery as Part 4. Patches are to
an image what tokens are to a sentence.

In [ ]:
from transformers import CLIPModel, CLIPProcessor

CLIP_ID = "openai/clip-vit-base-patch32"
vit = CLIPModel.from_pretrained(CLIP_ID, attn_implementation="eager").to(DEVICE).eval()
proc = CLIPProcessor.from_pretrained(CLIP_ID)

PICK = 0                                          # change this
img = Image.open(paths[PICK]).convert("RGB")
px = proc(images=img, return_tensors="pt")["pixel_values"]
square = np.array(img.resize((224, 224)))

fig, (a, b) = plt.subplots(1, 2, figsize=(9, 4.6))
a.imshow(square); a.set_title("what you see", loc="left", fontsize=10)
b.imshow(square)
for g in range(0, 225, 32):
    b.axvline(g, color="w", lw=1.2); b.axhline(g, color="w", lw=1.2)
b.set_title("what it sees: 49 patches", loc="left", fontsize=10)
for ax in (a, b):
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()
print(px.shape, "->", 224 // 32, "x", 224 // 32, "patches + 1 CLS token")

In [ ]:
# Every patch on its own. This is the whole input, in order.
patches = square.reshape(7, 32, 7, 32, 3).transpose(0, 2, 1, 3, 4)
fig, axes = plt.subplots(7, 7, figsize=(5.6, 5.6))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(patches[i // 7, i % 7]); ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("49 tokens", x=0.02, ha="left", fontsize=12)
plt.tight_layout(); plt.show()

### Where it looks

Attention says how much each token pulls from every other. The CLS token is the one that
becomes the image's vector, so its attention row is a map of what ended up mattering.

In [ ]:
with torch.no_grad():
    att = vit.vision_model(px.to(DEVICE), output_attentions=True).attentions

def cls_map(layer):
    a = att[layer][0].mean(0)[0, 1:]              # average the heads, CLS row, drop CLS itself
    return a.reshape(7, 7).float().cpu().numpy()

# Dim each patch by how little the CLS token looked at it.
fig, axes = plt.subplots(1, 4, figsize=(12, 3.4))
axes[0].imshow(square / 255); axes[0].set_title("image", loc="left", fontsize=10)
for ax, layer in zip(axes[1:], (0, 5, 11)):
    m = cls_map(layer)
    lit = np.kron(m / m.max(), np.ones((32, 32)))[..., None]
    ax.imshow(square / 255 * (0.15 + 0.85 * lit))
    ax.set_title(f"layer {layer}", loc="left", fontsize=10)
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("where the CLS token looked", x=0.02, ha="left", fontsize=12)
plt.tight_layout(); plt.show()

for layer in (0, 5, 11):
    m = cls_map(layer)
    print(f"layer {layer:2d}  busiest patch gets {m.max()/m.mean():.1f}x the average")

Layer 0 is nearly flat: every patch about the same. By layer 11 a few patches carry several
times the average, mostly on the face and the hat.

Do not over-read it. Attention shows where information was pulled from, not why. Treat a
picture like this as a hypothesis, not evidence. Jain and Wallace, *Attention is not
Explanation* (2019), is the argument.

## 7 · Train a CLIP

CLIP's training rule is one line: **the matching pair should score higher than every mismatched
pair in the batch.** Take a batch of N pictures and their N captions, score all N×N
combinations, and push the diagonal up. That is it. It is Week 4's contrastive rule with
pictures on one side.

Real CLIP needs 400 million pairs. The rule works on thirty, so you can watch it happen.

Data: coloured shapes and captions like *a red triangle*. Six of the thirty combinations are
held out — the model never sees a green circle, so we can ask whether it can put two words
together it never saw together.

In [ ]:
rng = np.random.default_rng(0)
torch.manual_seed(0)

COLOURS = {"red": (220, 60, 50), "blue": (60, 90, 210), "green": (60, 160, 80),
           "yellow": (235, 195, 50), "purple": (140, 70, 180), "pink": (230, 120, 170)}
SHAPES = ["circle", "square", "triangle", "cross", "diamond"]

def draw(colour, shape, size=32):
    img = np.full((size, size, 3), 245, np.float32)
    cy, cx = rng.integers(12, 21, 2)
    r = int(rng.integers(7, 10))
    yy, xx = np.mgrid[0:size, 0:size]
    dy, dx = yy - cy, xx - cx
    if shape == "circle":     mask = dy**2 + dx**2 <= r*r
    elif shape == "square":   mask = (abs(dy) <= r*.8) & (abs(dx) <= r*.8)
    elif shape == "triangle": mask = (dy <= r*.8) & (dy >= -r*.8) & (abs(dx) <= (r*.8 - dy)*.7)
    elif shape == "diamond":  mask = abs(dy) + abs(dx) <= r
    else:                     mask = (((abs(dy) <= r*.3) & (abs(dx) <= r)) |
                                      ((abs(dx) <= r*.3) & (abs(dy) <= r)))
    img[mask] = COLOURS[colour]
    return img / 255.0

ALL = [(c, s) for c in COLOURS for s in SHAPES]
HELD = {("green", "circle"), ("blue", "square"), ("red", "cross"),
        ("purple", "triangle"), ("pink", "diamond"), ("yellow", "circle")}
TRAIN = [p for p in ALL if p not in HELD]
WORDS = {w: i for i, w in enumerate(["a"] + list(COLOURS) + SHAPES)}

def make(combos):
    x = np.stack([draw(c, s) for c, s in combos]).transpose(0, 3, 1, 2)
    t = [[WORDS["a"], WORDS[c], WORDS[s]] for c, s in combos]
    return torch.tensor(x, dtype=torch.float32), torch.tensor(t)

fig, axes = plt.subplots(1, 6, figsize=(9, 1.8))
for ax, (c, s) in zip(axes, TRAIN[:6]):
    ax.imshow(draw(c, s)); ax.set_title(f"a {c} {s}", fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()
print(len(TRAIN), "combinations to train on,", len(HELD), "held back")

In [ ]:
class PictureSide(nn.Module):
    def __init__(self, dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 16, 3, 2, 1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, 2, 1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, 2, 1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, dim))
    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)

class WordSide(nn.Module):
    def __init__(self, dim=64):
        super().__init__()
        self.embed = nn.Embedding(len(WORDS), dim)
        self.out = nn.Linear(dim, dim)
    def forward(self, t):
        return F.normalize(self.out(self.embed(t).mean(1)), dim=-1)

eyes, ears = PictureSide(), WordSide()
temperature = nn.Parameter(torch.tensor(2.5))
opt = torch.optim.Adam(list(eyes.parameters()) + list(ears.parameters()) + [temperature], lr=3e-3)

losses, first = [], None
for step in range(800):
    x, t = make(TRAIN)                                   # fresh images, same 24 captions
    scores = eyes(x) @ ears(t).T * temperature.exp()     # 24 x 24, every combination
    gold = torch.arange(len(x))                          # the right answer is the diagonal
    loss = (F.cross_entropy(scores, gold) + F.cross_entropy(scores.T, gold)) / 2
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())
    if first is None:
        first = scores.detach().numpy()

print(f"start {losses[0]:.2f}   end {losses[-1]:.3f}   (guessing would be {np.log(len(TRAIN)):.2f})")

In [ ]:
with torch.no_grad():
    x, t = make(TRAIN)
    last = (eyes(x) @ ears(t).T * temperature.exp()).numpy()

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, M, name in zip(axes, (first, last), ("before training", "after training")):
    ax.imshow(M, cmap="RdYlBu_r")
    ax.set_title(name, loc="left", fontsize=10)
    ax.set_xlabel("captions"); ax.set_ylabel("pictures")
    ax.set_xticks([]); ax.set_yticks([])
axes[2].plot(losses, color="#A34526", lw=1.4)
axes[2].set_title("loss", loc="left", fontsize=10)
axes[2].set_xlabel("step")
plt.tight_layout(); plt.show()

Before training the scores depend only on the caption, so the panel is vertical stripes. After
training the diagonal is the brightest thing in it. That is the whole of contrastive learning:
lift the diagonal, push down everything else.

Now the held-out combinations, which it never saw.

In [ ]:
eyes.eval(); ears.eval()
with torch.no_grad():
    x, t = make(ALL)
    S = (eyes(x) @ ears(t).T).numpy()

print(f"all {len(ALL)} combinations, picture picks its own caption: "
      f"{(S.argmax(1) == np.arange(len(ALL))).mean():.0%}\n")
for i, pair in enumerate(ALL):
    if pair in HELD:
        got = ALL[int(S[i].argmax())]
        mark = "yes" if got == pair else " no"
        print(f"  {mark}   {'a ' + pair[0] + ' ' + pair[1]:22s} -> a {got[0]} {got[1]}")

Colour it always gets. Shape it mostly gets, and the misses are between shapes that fill
about the same area. Colour is one strong signal; shape needs the convolutions to work harder,
and 24 combinations is not much to learn from.

What matters is that it composes at all. Nothing in training paired *green* with *circle*, and
the two words still land in the right place together.

In [ ]:
# Your turn: add a shape, hold out a different set, shrink the network, change lr.
# Or swap `draw` for real images and captions of your own.

## Next

- **Your own corpus.** Part 5 works on any list of strings. Swap in the CSV from Week 4.
- **A bigger embedder.** `all-mpnet-base-v2` is slower and better. The
  [MTEB leaderboard](https://huggingface.co/spaces/mteb/leaderboard) ranks the rest.
- **Fine-tuning.** `cool-methods/finetune_modernbert.ipynb`. Do it only after an off-the-shelf
  model has clearly failed you.
- **Other pipelines.** `"zero-shot-classification"`, `"ner"`, `"summarization"`,
  `"image-classification"`. Same three lines, different task name.

Three things worth repeating:

1. Name the model and version in anything you write. "An AI said" is not a method.
2. A model trained on the open web knows the open web. Your corpus is not that.
3. A confident score is not a correct answer, and the model cannot tell you which you have.